# Practical Application III: Comparing Classifiers

**Overview**: In this practical application, your goal is to compare the performance of the classifiers we encountered in this section, namely K Nearest Neighbor, Logistic Regression, Decision Trees, and Support Vector Machines.  We will utilize a dataset related to marketing bank products over the telephone.  



### Getting Started

Our dataset comes from the UCI Machine Learning repository [link](https://archive.ics.uci.edu/ml/datasets/bank+marketing).  The data is from a Portugese banking institution and is a collection of the results of multiple marketing campaigns.  We will make use of the article accompanying the dataset [here](CRISP-DM-BANK.pdf) for more information on the data and features.



### Problem 1: Understanding the Data

To gain a better understanding of the data, please read the information provided in the UCI link above, and examine the **Materials and Methods** section of the paper.  How many marketing campaigns does this data represent?

#### According to the Materials and Methods section of the paper, the dataset was collected from 17 separate direct marketing campaigns conducted by a Portuguese bank. These campaigns took place over 2.5 years, between May 2008 and Nov 2010

### Problem 2: Read in the Data

Use pandas to read in the dataset `bank-additional-full.csv` and assign to a meaningful variable name.

In [9]:
import pandas as pd

In [10]:
df = pd.read_csv('data/bank-additional-full.csv', sep = ';')

In [11]:
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


### Problem 3: Understanding the Features


Examine the data description below, and determine if any of the features are missing values or need to be coerced to a different data type.


```
Input variables:
# bank client data:
1 - age (numeric)
2 - job : type of job (categorical: 'admin.','blue-collar','entrepreneur','housemaid','management','retired','self-employed','services','student','technician','unemployed','unknown')
3 - marital : marital status (categorical: 'divorced','married','single','unknown'; note: 'divorced' means divorced or widowed)
4 - education (categorical: 'basic.4y','basic.6y','basic.9y','high.school','illiterate','professional.course','university.degree','unknown')
5 - default: has credit in default? (categorical: 'no','yes','unknown')
6 - housing: has housing loan? (categorical: 'no','yes','unknown')
7 - loan: has personal loan? (categorical: 'no','yes','unknown')
# related with the last contact of the current campaign:
8 - contact: contact communication type (categorical: 'cellular','telephone')
9 - month: last contact month of year (categorical: 'jan', 'feb', 'mar', ..., 'nov', 'dec')
10 - day_of_week: last contact day of the week (categorical: 'mon','tue','wed','thu','fri')
11 - duration: last contact duration, in seconds (numeric). Important note: this attribute highly affects the output target (e.g., if duration=0 then y='no'). Yet, the duration is not known before a call is performed. Also, after the end of the call y is obviously known. Thus, this input should only be included for benchmark purposes and should be discarded if the intention is to have a realistic predictive model.
# other attributes:
12 - campaign: number of contacts performed during this campaign and for this client (numeric, includes last contact)
13 - pdays: number of days that passed by after the client was last contacted from a previous campaign (numeric; 999 means client was not previously contacted)
14 - previous: number of contacts performed before this campaign and for this client (numeric)
15 - poutcome: outcome of the previous marketing campaign (categorical: 'failure','nonexistent','success')
# social and economic context attributes
16 - emp.var.rate: employment variation rate - quarterly indicator (numeric)
17 - cons.price.idx: consumer price index - monthly indicator (numeric)
18 - cons.conf.idx: consumer confidence index - monthly indicator (numeric)
19 - euribor3m: euribor 3 month rate - daily indicator (numeric)
20 - nr.employed: number of employees - quarterly indicator (numeric)

Output variable (desired target):
21 - y - has the client subscribed a term deposit? (binary: 'yes','no')
```



In [12]:
df.isna().sum()

age               0
job               0
marital           0
education         0
default           0
housing           0
loan              0
contact           0
month             0
day_of_week       0
duration          0
campaign          0
pdays             0
previous          0
poutcome          0
emp.var.rate      0
cons.price.idx    0
cons.conf.idx     0
euribor3m         0
nr.employed       0
y                 0
dtype: int64

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  object 
 2   marital         41188 non-null  object 
 3   education       41188 non-null  object 
 4   default         41188 non-null  object 
 5   housing         41188 non-null  object 
 6   loan            41188 non-null  object 
 7   contact         41188 non-null  object 
 8   month           41188 non-null  object 
 9   day_of_week     41188 non-null  object 
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  object 
 15  emp.var.rate    41188 non-null  float64
 16  cons.price.idx  41188 non-null  float64
 17  cons.conf.idx   41188 non-null 

In [15]:
df.describe()

,age,duration,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed
count,41188.00000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000
mean,40.02406,258.285010,2.567593,962.475454,0.172963,0.081886,93.575664,-40.502600,3.621291,5167.035911
std,10.42125,259.279249,2.770014,186.910907,0.494901,1.570960,0.578840,4.628198,1.734447,72.251528
min,17.00000,0.000000,1.000000,0.000000,0.000000,-3.400000,92.201000,-50.800000,0.634000,4963.600000
25%,32.00000,102.000000,1.000000,999.000000,0.000000,-1.800000,93.075000,-42.700000,1.344000,5099.100000
50%,38.00000,180.000000,2.000000,999.000000,0.000000,1.100000,93.749000,-41.800000,4.857000,5191.000000
75%,47.00000,319.000000,3.000000,999.000000,0.000000,1.400000,93.994000,-36.400000,4.961000,5228.100000
max,98.00000,4918.000000,56.000000,999.000000,7.000000,1.400000,94.767000,-26.900000,5.045000,5228.100000


### Technically speaking, there are no missing values in any of the columns.

However, there are few scenarios to deal with:
1) Some of the categorical values have "unknown" as value (example: job, marital, education, default, etc) which may in itself explain some charateristics of the particular sample. So handling them is key in the following steps
2) Special value 999 for pdays - need to handle this to make sure the context of this variable is not lost
3) Values can mean different things - poutcome has value "nonexistent' which doesn't mean it is missing value
4) duration variable cannot be used for prediction (as explained in the notes itself, this variable can be observed before the event/outcome)


### Problem 4: Understanding the Task

After examining the description and data, your goal now is to clearly state the *Business Objective* of the task.  State the objective below.

## Business Objective: The objective was to reduce marketing costs while maintaining deposit subscriptions.

### Problem 5: Engineering Features

Now that you understand your business objective, we will build a basic model to get started.  Before we can do this, we must work to encode the data.  Using just the bank information features, prepare the features and target column for modeling with appropriate encoding and transformations.

In [17]:
#standard imports

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline


# Features using only bank client information
bank_features = [
    'age',
    'job',
    'marital',
    'education',
    'default',
    'housing',
    'loan'
]

#assing X and Y variables
X = df[bank_features]
y = df['y'].map({'no': 0, 'yes': 1})

# Separate numeric and categorical features

numeric_features = ['age']

categorical_features = [
    'job',
    'marital',
    'education',
    'default',
    'housing',
    'loan'
]

# Preprocessing:
# scale numeric feature
# one-hot encode categorical features

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
    ]
)


### Problem 6: Train/Test Split

With your data prepared, split it into a train and test set.

In [22]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)


In [24]:
# Fit the preprocessing transformer only on training data
X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)


In [25]:
# Check shapes
print("X_train shape:", X_train_encoded.shape)
print("X_test shape:", X_test_encoded.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)


X_train shape: (30891, 28)
X_test shape: (10297, 28)
y_train shape: (30891,)
y_test shape: (10297,)


### Problem 7: A Baseline Model

Before we build our first model, we want to establish a baseline.  What is the baseline performance that our classifier should aim to beat?

In [28]:
from sklearn.dummy import DummyClassifier

# Majority class baseline
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)
baseline_score = dummy.score(X_test, y_test)

print(f"Baseline Accuracy: {baseline_score:.4f}")

Baseline Accuracy: 0.8873


In [29]:
# alternate method to get baseline

baseline = (y == 0).mean()
print(f"Baseline Accuracy: {baseline:.4f}")

Baseline Accuracy: 0.8873


### This banking marketing dataset is highly imbalanced with 88.7% "No" and  11.3% "Yes" response to the campaign. So, baseline accuracy alone of 88.7% isn't if everyone who responds ends being "no". The business objective will not be met. 

### Problem 8: A Simple Model

Use Logistic Regression to build a basic model on your data.  

In [35]:
from sklearn.linear_model import LogisticRegression

# Build the pipeline
logreg_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000, random_state=42))
])

# Fit the model
logreg_pipe.fit(X_train, y_train)



,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


### Problem 9: Score the Model

What is the accuracy of your model?

In [37]:
# Evaluate on the test set
logreg_score = logreg_pipe.score(X_test, y_test)

print(f"Logistic Regression Accuracy: {logreg_score:.4f}")

Logistic Regression Accuracy: 0.8873


### The basic logistic model accuracy is 0.8873, the same as the basic model

### Problem 10: Model Comparisons

Now, we aim to compare the performance of the Logistic Regression model to our KNN algorithm, Decision Tree, and SVM models.  Using the default settings for each of the models, fit and score each.  Also, be sure to compare the fit time of each of the models.  Present your findings in a `DataFrame` similar to that below:

| Model | Train Time | Train Accuracy | Test Accuracy |
| ----- | ---------- | -------------  | -----------   |
|     |    |.     |.     |

In [51]:
import time
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Preprocessor for models that benefit from scaling ==> already implemented previously

# Preprocessor for tree model, no scaling needed
preprocessor_unscaled = ColumnTransformer([
    ('num', 'passthrough', numeric_features),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
])


models = {
    "Logistic Regression": Pipeline([
        ("preprocess", preprocessor),
        ("model", LogisticRegression(max_iter=1000))
    ]),
    
    "KNN": Pipeline([
        ("preprocess", preprocessor),
        ("model", KNeighborsClassifier())
    ]),
    
    "Decision Tree": Pipeline([
        ("preprocess", preprocessor_unscaled),
        ("model", DecisionTreeClassifier(random_state=42))
    ]),
    
    "SVM": Pipeline([
        ("preprocess", preprocessor),
        ("model", SVC(probability=True))
    ])
}



In [54]:
results = []

for name, model in models.items():
    start_time = time.time()
    model.fit(X_train, y_train)
    fit_time = time.time() - start_time
    
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, y_proba),
        "Fit_Time_Seconds": fit_time
    })

In [55]:
model_comparison = (
    pd.DataFrame(results)
    .sort_values(by="F1", ascending=False)
    .reset_index(drop=True)
)

model_comparison

,Model,Accuracy,Precision,Recall,F1,ROC_AUC,Fit_Time_Seconds
0,KNN,0.878314,0.337979,0.083621,0.134070,0.586903,0.033987
1,Decision Tree,0.865592,0.238318,0.087931,0.128463,0.577890,0.141666
2,SVM,0.886957,0.441176,0.012931,0.025126,0.567951,104.705940
3,Logistic Regression,0.887346,0.000000,0.000000,0.000000,0.655824,0.088213


### Basline model summary

These baseline models establish that customer demographic and financial profile alone is insufficient to accurately which contact will lead to a deposit:
* Logistic Regression achieves the highest accuracy (88.7%), but does so by predicting every customer as a non-subscriber, resulting in zero precision, recall, and F1-score.
* KNN and Decision Tree identify a small number of positive cases and therefore achieve higher recall and F1-scores, albeit at the cost of slightly lower accuracy.
* SVM performs poorly in terms of recall while requiring by far the longest training time (over 100 seconds), making it the least attractive model in this baseline comparison.


Overall, the results indicate that richer information—such as previous marketing interactions, contact characteristics, and macroeconomic variables—is likely required before these classifiers can meaningfully distinguish customers who will subscribe from those who will not.

### Problem 11: Improving the Model

Now that we have some basic models on the board, we want to try to improve these.  Below, we list a few things to explore in this pursuit.


- Hyperparameter tuning and grid search.  All of our models have additional hyperparameters to tune and explore.  For example the number of neighbors in KNN or the maximum depth of a Decision Tree.  
- Adjust your performance metric

In [56]:
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_validate, StratifiedKFold

In [59]:
# Helper Function
# Fits a model, measures training time, evaluates performance, and returns all metrics as a dictionary.
# This helper will be reused throughout the notebook for:
# baseline models
# hyperparameter tuning
# class balancing
# feature engineering experiments

def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    
    # Measure training time
    start_time = time.time()
    model.fit(X_train, y_train)
    fit_time = time.time() - start_time

    # Predictions
    y_pred = model.predict(X_test)

    # probability estimates used for ROC-AUC
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
    else:
        y_proba = model.decision_function(X_test)

    return {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, y_proba),
        "Fit_Time_Seconds": fit_time
    }


In [67]:
# Setting up hyperparamets for all 4 techniques to be used in grid search

models_and_params = {
    
    "Logistic Regression": {
        "pipeline": Pipeline([
            ("preprocess", preprocessor),
            ("model", LogisticRegression(max_iter=1000))
        ]),
        "params": {
            "model__C": [0.01, 0.1, 1, 10],
            "model__solver": ["liblinear", "lbfgs"]
        }
    },

    "KNN": {
        "pipeline": Pipeline([
            ("preprocess", preprocessor),
            ("model", KNeighborsClassifier())
        ]),
        "params": {
            "model__n_neighbors": [3, 5, 7, 11, 15],
            "model__weights": ["uniform", "distance"],
            "model__p": [1, 2]
        }
    },

    "Decision Tree": {
        "pipeline": Pipeline([
            ("preprocess", preprocessor_unscaled),
            ("model", DecisionTreeClassifier(random_state=42))
        ]),
        "params": {
            "model__max_depth": [2, 3, 5, 10, None],
            "model__min_samples_split": [2, 10, 50],
            "model__min_samples_leaf": [1, 5, 20],
            "model__criterion": ["gini", "entropy"]
        }
    },

    "SVM": {
        "pipeline": Pipeline([
            ("preprocess", preprocessor),
            ("model", SVC(probability=True))
        ]),
        "params": {
            "model__C": [0.1, 1, 10],
            "model__kernel": ["linear", "rbf"],
            "model__gamma": ["scale", "auto"]
        }
    }
}



In [69]:
# Grid search to find best parameters for each model

grid_results = []
best_models = {}

for name, item in models_and_params.items():
    
    # just a visual tracking
    print(f"Running GridSearchCV for {name}...")
    
    start_time = time.time()
    
    # define the grid
    grid = GridSearchCV(
        estimator=item["pipeline"],
        param_grid=item["params"],
        scoring="f1",
        cv=3,
        n_jobs=-1
    )

    # run the grid search 
    grid.fit(X_train, y_train)
    
    grid_time = time.time() - start_time
    
    best_models[name] = grid.best_estimator_
    
    test_result = evaluate_model(
        name=name,
        model=grid.best_estimator_,
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test
    )
    
    test_result["Best_Params"] = grid.best_params_
    test_result["Best_CV_F1"] = grid.best_score_
    test_result["GridSearch_Time_Seconds"] = grid_time

    # Track grid results
    grid_results.append(test_result)

# Create a dataframe to store results
grid_results_df = (
    pd.DataFrame(grid_results)
    .sort_values("F1", ascending=False)
    .reset_index(drop=True)
)


grid_results_df


Running GridSearchCV for Logistic Regression...
Running GridSearchCV for KNN...


/opt/anaconda3/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Running GridSearchCV for Decision Tree...


/opt/anaconda3/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Running GridSearchCV for SVM...


/opt/anaconda3/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


,Model,Accuracy,Precision,Recall,F1,ROC_AUC,Fit_Time_Seconds,Best_Params,Best_CV_F1,GridSearch_Time_Seconds
0,KNN,0.861319,0.230924,0.099138,0.138721,0.557880,0.036780,"{'model__n_neighbors': 3, 'model__p': 1, 'mode...",0.152934,26.827862
1,Decision Tree,0.865592,0.238318,0.087931,0.128463,0.577890,0.157034,"{'model__criterion': 'gini', 'model__max_depth...",0.142316,2.638925
2,SVM,0.885695,0.369231,0.020690,0.039184,0.570705,247.306414,"{'model__C': 10, 'model__gamma': 'scale', 'mod...",0.051140,592.810254
3,Logistic Regression,0.887346,0.000000,0.000000,0.000000,0.645689,0.042482,"{'model__C': 0.01, 'model__solver': 'liblinear'}",0.000000,2.496612


### Hyperparameter tuning provided a very small improvements:

* KNN improved slightly: F1 went from about 0.134 → 0.139
* Decision Tree stayed basically the same
* SVM became even slower: fit time around 247 sec, grid search around 593 sec
* Logistic Regression still failed to predict positives, even though ROC-AUC is still the best among the models (at the current basic 0.5 threshold)

In [72]:
# Class imbalance handling

balanced_models = {    
    "Logistic Regression Balanced": Pipeline([
        ("preprocess", preprocessor),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced"
        ))
    ]),
    
    "Decision Tree Balanced": Pipeline([
        ("preprocess", preprocessor_unscaled),
        ("model", DecisionTreeClassifier(
            random_state=42,
            class_weight="balanced"
        ))
    ]),
    
    "KNN": Pipeline([
        ("preprocess", preprocessor),
        ("model", KNeighborsClassifier())
    ]),
    
    "SVM Balanced": Pipeline([
        ("preprocess", preprocessor),
        ("model", SVC(
            probability=True,
            class_weight="balanced"
        ))
    ])
}



In [74]:

# run balanced models

imbalance_results = []
for name, model in balanced_models.items():
    result = evaluate_model(
        name=name,
        model=model,
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test
    )
    
    imbalance_results.append(result)

# store results in a dataframe

imbalance_results_df = (
    pd.DataFrame(imbalance_results)
    .sort_values("F1", ascending=False)
    .reset_index(drop=True)
)

imbalance_results_df



,Model,Accuracy,Precision,Recall,F1,ROC_AUC,Fit_Time_Seconds
0,SVM Balanced,0.619307,0.166183,0.592241,0.259539,0.649962,114.056780
1,Logistic Regression Balanced,0.595610,0.163229,0.627586,0.259075,0.656581,0.097330
2,Decision Tree Balanced,0.678062,0.154537,0.415517,0.225286,0.577607,0.154447
3,KNN,0.878314,0.337979,0.083621,0.134070,0.586903,0.027453


### Class imbalance handling has made SVM and Logistic Regression more relevant

Applying class_weight="balanced" transformed Logistic Regression and SVM from models that almost never identified subscribers into models that correctly detected approximately 60% of the positive cases. Although this came at the cost of lower overall accuracy, the balanced models are far more valuable for a direct marketing application, where the primary goal is to identify potential subscribers rather than simply maximize accuracy. Also, SVM is resource intensive r


##### Questions